<a href="https://colab.research.google.com/github/RisalKalwar/Starter-Notebooks/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RisalKalwar/Starter-Notebooks/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [ ]:
## 1. My lane

**Lane: Performance-based opportunity modeling** (provisional — can revisit by end of Week 4).

I'm choosing this lane because I've already run the starter pipeline on `content_refresh_anonymized.csv`
and found three real patterns in the data (below) that all point toward the same kind of decision:
which existing pages are worth fixing first. That's a natural fit for ranking content actions by
expected impact rather than starting a new problem from scratch.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [ ]:
## 2. The question

**Search question:** Given a set of existing indexed pages, which ones should be prioritized for
content refresh to recover lost search performance?

**Unit of analysis:** a page (one row = one URL/page, observed over a 90-day window).

**Output:** a ranked list (queue) of pages, ordered by predicted likelihood of being a
high-value refresh candidate — i.e. pages that are declining but still have recoverable traffic potential.

**The decision it improves:** which pages a content team spends limited hours rewriting/refreshing
this month, instead of guessing from gut feel or an outdated "refresh everything over 12 months old" rule.

**The action someone takes:** a content strategist pulls the top N pages off the ranked queue and
assigns them for rewrite/refresh work.

**Cost of a wrong recommendation:**
- False positive (model flags a page that isn't actually worth refreshing) → wasted writer/editor hours,
  opportunity cost of not refreshing a page that actually needed it.
- False negative (model misses a page that's quietly declining) → continued organic traffic loss,
  compounding over time before anyone notices.

**Why data/ML can help at all:** the hand-written rule baseline in the starter pipeline only reaches
~24% precision at picking the right pages — meaning 3 out of 4 "priority" pages it flags are not
actually worth fixing first. A model can pick up non-obvious combinations of signals (position, CTR,
engagement, trend) that a single hand-written threshold rule can't capture, which is why the learned
model in the pipeline reached roughly 3x that precision.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

def find_repo_root():
    """Walk up until we find data/raw, or return None."""
    cwd = os.getcwd()
    while True:
        if os.path.isdir(os.path.join(cwd, "data", "raw")):
            return cwd
        parent = os.path.dirname(cwd)
        if parent == cwd:
            return None
        cwd = parent

root = find_repo_root()

if root is None:
    if IN_COLAB:
        os.chdir("/content")
        if not os.path.isdir(REPO_DIR):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        os.chdir(REPO_DIR)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    else:
        raise RuntimeError("Could not locate repo root — run from inside the repo.")
else:
    os.chdir(root)

print("Working dir:", os.getcwd())

# Make sure the pipeline outputs exist — regenerate if missing
if not os.path.exists("outputs/model_results.json"):
    print("outputs/model_results.json missing — running the pipeline now...")
    subprocess.run([sys.executable, "scripts/run_all.py"], check=True)

import pandas as pd, numpy as np, json

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")

decline_share = (df["trend_direction"] == "down").mean()
print(f"Share of pages currently declining: {decline_share:.1%}")

visible = df[df["impressions_90d"] >= 100]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print("\nCTR by position tier:")
print(ctr_by_pos.round(4).to_string())

res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]
print(f"\nBaseline rule Precision@50: {base:.3f}")
print(f"Random forest Precision@50: {rf:.3f}")
print(f"Model beats the hand-written rule by ~{rf/base:.1f}x — this is the gap this lane tries to close.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
outputs/model_results.json missing — running the pipeline now...
30000 rows, 44 columns
Share of pages currently declining: 54.2%

CTR by position tier:
position_tier
page_1      0.3548
top_3       0.3341
striking    0.2558
page_3_5    0.1424
deep        0.0554

Baseline rule Precision@50: 0.240
Random forest Precision@50: 0.740
Model beats the hand-written rule by ~3.1x — this is the gap this lane tries to close.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [ ]:

- I can claim these patterns are **observed in this anonymized sample** — not proven causal relationships.
- I cannot claim I've "reverse-engineered Google's algorithm" — CTR and position correlate, but I'm not
  modeling search engine ranking logic itself.
- Precision@50 numbers reflect **this sample and this split**; they may shift with more data or a
  different time window.
- Any recommendation this lane produces is **decision-support**, not an autonomous action — a human
  still decides what to refresh.
- I will not use any client-identifying data in any output I publish, per `DATA_USE.md`.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.